# Day 1: PyTorch 训练循环 — 从零到 IMDb 情感分类

**目标**：彻底理解深度学习训练的核心机制，为后续 LLM 微调打下基础。

作为 AI/Agent 工程师，你可能每天都在调用 `model.fit()` 或 HuggingFace `Trainer`，但当模型行为异常时——loss 不降、梯度爆炸、过拟合——你需要**深入训练循环内部**才能诊断问题。

## 为什么 Agent 工程师需要理解训练流程？

| 场景 | 需要的知识 |
|------|------------|
| 微调 LLM 做工具调用 | 理解 loss 曲线，判断是否过拟合 |
| 部署模型到生产环境 | 理解 `model.eval()` vs `model.train()` 的区别 |
| Debug Agent 输出质量下降 | 检查梯度、学习率、数据分布 |
| 选择合适的微调策略 | 理解 optimizer、学习率调度的影响 |

本 notebook 将带你**手写每一步**，不用任何高级封装。

## 本节内容概览

1. **训练循环五步曲**：Forward → Loss → Backward → Step → Repeat
2. **Self-Attention 原理**：手动计算注意力矩阵
3. **IMDb 实战**：用 Transformer 做情感分类
4. **总结与思考题**

In [ ]:
# 基础依赖
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# 检查 GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {DEVICE}')
print(f'PyTorch 版本: {torch.__version__}')

---
# Part 1: 训练循环五步曲

深度学习训练的本质就是**反复调整参数使 loss 下降**。每一步包含：

```
Forward Pass → 计算 Loss → Backward Pass → Optimizer Step → 清零梯度
```

我们先用最简单的 2 层全连接网络来演示。

## Step 1: Forward Pass（前向传播）

数据通过网络，逐层计算输出。每一层做的事情：`output = activation(W @ input + b)`

In [ ]:
# 构建一个最简单的 2 层全连接网络
class SimpleNet(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8, output_dim=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)   # 第一层: 4 -> 8
        self.fc2 = nn.Linear(hidden_dim, output_dim)   # 第二层: 8 -> 3
    
    def forward(self, x):
        x = F.relu(self.fc1(x))  # ReLU 激活
        x = self.fc2(x)          # 输出层（不加激活，交给 CrossEntropy）
        return x

# 创建模型实例
model = SimpleNet()
print('模型结构:')
print(model)
print(f'\n参数量: {sum(p.numel() for p in model.parameters())}')

In [ ]:
# 造一些假数据来演示 forward pass
x_sample = torch.randn(2, 4)  # 2 个样本，每个 4 维特征
print('输入 x:')
print(x_sample)

# Forward pass：数据流过网络
output = model(x_sample)
print('\n输出 logits（未经 softmax 的原始分数）:')
print(output)
print(f'\n输出形状: {output.shape}  (2个样本, 3个类别)')

# 转换为概率
probs = F.softmax(output, dim=-1)
print(f'\n转换为概率:')
print(probs)
print(f'每行概率之和: {probs.sum(dim=-1)}')

## Step 2: Loss 计算

**Loss（损失函数）** 衡量模型预测和真实标签之间的差距。

- **CrossEntropyLoss**：分类任务的标准 loss，内部 = `log_softmax` + `NLLLoss`
- 直觉：如果真实类别的预测概率越高，loss 越低

In [ ]:
# 假设真实标签是 [0, 2]（第一个样本属于类别0，第二个属于类别2）
labels = torch.tensor([0, 2])

# 计算 CrossEntropy Loss
criterion = nn.CrossEntropyLoss()
loss = criterion(output, labels)
print(f'CrossEntropy Loss: {loss.item():.4f}')

# 手动验证：CrossEntropy = -log(正确类别的概率)
manual_loss = -torch.log(probs[0, 0]) - torch.log(probs[1, 2])
manual_loss = manual_loss / 2  # 取平均
print(f'手动计算 Loss:    {manual_loss.item():.4f}')
print('两者应该非常接近（浮点误差可忽略）')

## Step 3: Backward Pass（反向传播）

调用 `loss.backward()` 后，PyTorch 自动计算每个参数的梯度（`∂loss/∂param`）。

这就是**链式法则**的应用：从 loss 出发，沿计算图反向传播，算出每个权重对 loss 的贡献。

In [ ]:
# 反向传播前，梯度为 None
print('backward 之前:')
print(f'  fc1.weight.grad = {model.fc1.weight.grad}')

# 执行反向传播
loss.backward()

# 反向传播后，每个参数都有了梯度
print('\nbackward 之后:')
print(f'  fc1.weight.grad 形状: {model.fc1.weight.grad.shape}')
print(f'  fc1.weight.grad 值（前2行）:\n{model.fc1.weight.grad[:2]}')
print(f'\n  fc2.weight.grad 形状: {model.fc2.weight.grad.shape}')
print(f'  fc2.weight.grad 值:\n{model.fc2.weight.grad}')

## Step 4: Optimizer Step（参数更新）

有了梯度，就可以更新参数了。最基本的方法是 **SGD**：

$$\theta_{new} = \theta_{old} - lr \times \nabla_{\theta} Loss$$

**Adam** 更智能：它会根据梯度的历史均值和方差来自适应调整每个参数的学习率。

In [ ]:
# 对比 SGD 和 Adam
# 先记录当前权重
old_weight = model.fc1.weight.data.clone()

# 手动 SGD 更新（lr=0.01）
lr = 0.01
manual_update = old_weight - lr * model.fc1.weight.grad
print('手动 SGD 更新后的权重（前2行）:')
print(manual_update[:2])

# 用 PyTorch 的 SGD optimizer
optimizer_sgd = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer_sgd.step()  # 执行一步更新
print('\nPyTorch SGD 更新后的权重（前2行）:')
print(model.fc1.weight.data[:2])

print('\n两者完全一致！SGD 就是这么简单。')

In [ ]:
# Adam vs SGD 对比实验
# 在随机数据上训练，观察收敛速度差异

torch.manual_seed(42)
X_train = torch.randn(200, 4)  # 200 个样本
y_train = torch.randint(0, 3, (200,))  # 3 分类

def train_with_optimizer(opt_name, lr, epochs=100):
    """用指定 optimizer 训练并返回 loss 历史"""
    torch.manual_seed(42)
    net = SimpleNet()
    if opt_name == 'SGD':
        opt = torch.optim.SGD(net.parameters(), lr=lr)
    else:
        opt = torch.optim.Adam(net.parameters(), lr=lr)
    
    losses = []
    for epoch in range(epochs):
        logits = net(X_train)           # Forward
        loss = criterion(logits, y_train)  # Loss
        opt.zero_grad()                 # 清零梯度
        loss.backward()                 # Backward
        opt.step()                      # Update
        losses.append(loss.item())
    return losses

losses_sgd = train_with_optimizer('SGD', lr=0.01)
losses_adam = train_with_optimizer('Adam', lr=0.01)

plt.figure(figsize=(8, 4))
plt.plot(losses_sgd, label='SGD (lr=0.01)', alpha=0.8)
plt.plot(losses_adam, label='Adam (lr=0.01)', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('SGD vs Adam 收敛速度对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f'SGD 最终 loss: {losses_sgd[-1]:.4f}')
print(f'Adam 最终 loss: {losses_adam[-1]:.4f}')

## Step 5: 完整 Mini Training Loop

把以上五步串起来，在随机数据上跑 100 个 epoch，画 loss 曲线。

```python
for epoch in range(EPOCHS):
    logits = model(x)          # 1. Forward
    loss = criterion(logits, y) # 2. Loss
    optimizer.zero_grad()       # 3. 清零梯度（重要！）
    loss.backward()             # 4. Backward
    optimizer.step()            # 5. Update
```

In [ ]:
# 完整训练循环演示
torch.manual_seed(0)
X = torch.randn(500, 4)     # 500 个样本，4 维特征
y = torch.randint(0, 3, (500,))  # 3 分类标签

model_demo = SimpleNet()
optimizer = torch.optim.Adam(model_demo.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

loss_history = []
acc_history = []

for epoch in range(100):
    # === 训练五步曲 ===
    logits = model_demo(X)              # 1. Forward Pass
    loss = criterion(logits, y)          # 2. 计算 Loss
    optimizer.zero_grad()                # 3. 清零梯度
    loss.backward()                      # 4. Backward Pass
    optimizer.step()                     # 5. 参数更新
    
    # 记录指标
    loss_history.append(loss.item())
    preds = logits.argmax(dim=-1)
    acc = (preds == y).float().mean().item()
    acc_history.append(acc)
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Acc: {acc*100:.1f}%')

In [ ]:
# 可视化训练过程
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(loss_history, color='steelblue')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('训练 Loss 曲线')
ax1.grid(True, alpha=0.3)

ax2.plot(acc_history, color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('训练准确率曲线')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('可以看到 loss 持续下降，准确率持续上升 — 模型在学习！')

### 关键注意事项

| 常见错误 | 后果 | 解决方法 |
|----------|------|----------|
| 忘记 `optimizer.zero_grad()` | 梯度累积，训练发散 | 每步都要清零 |
| 忘记 `loss.backward()` | 梯度为零，参数不更新 | 必须先 backward 再 step |
| 推理时忘记 `model.eval()` | Dropout/BatchNorm 行为错误 | 推理前切 eval 模式 |
| 学习率太大 | loss 爆炸 | 从 1e-3 或 1e-4 开始 |

---
# Part 2: Self-Attention 原理

Transformer 的核心是 **Self-Attention（自注意力）**。它让每个 token 能够"看到"序列中的其他 token，并根据相关性分配不同的注意力权重。

**核心公式**：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [ ]:
# 手动计算一个 3x3 的注意力矩阵
# 假设有 3 个 token，每个用 4 维向量表示
torch.manual_seed(42)

# 3 个 token 的嵌入向量
X = torch.randn(3, 4)  # (seq_len=3, d_model=4)
print('输入嵌入 X (3个token, 4维):')
print(X)

# Q, K, V 投影矩阵
W_q = torch.randn(4, 4)
W_k = torch.randn(4, 4)
W_v = torch.randn(4, 4)

# 计算 Q, K, V
Q = X @ W_q  # (3, 4)
K = X @ W_k  # (3, 4)
V = X @ W_v  # (3, 4)

print(f'\nQ 形状: {Q.shape}, K 形状: {K.shape}, V 形状: {V.shape}')

In [ ]:
import math

# 计算注意力分数: QK^T / sqrt(d_k)
d_k = Q.shape[-1]
attn_scores = (Q @ K.T) / math.sqrt(d_k)  # (3, 3)
print('注意力分数矩阵 (QK^T / sqrt(d_k)):')
print(attn_scores)

# Softmax 归一化 → 注意力权重
attn_weights = F.softmax(attn_scores, dim=-1)  # 每行和为 1
print(f'\n注意力权重矩阵 (softmax 后):')
print(attn_weights)
print(f'每行之和: {attn_weights.sum(dim=-1)}')

# 加权求和 → 输出
output = attn_weights @ V  # (3, 4)
print(f'\n输出 (attention_weights @ V):')
print(output)

In [ ]:
# 可视化注意力权重矩阵
fig, ax = plt.subplots(figsize=(5, 4))
tokens = ['Token 0', 'Token 1', 'Token 2']

im = ax.imshow(attn_weights.detach().numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_xlabel('Key (被关注的 token)')
ax.set_ylabel('Query (发起关注的 token)')
ax.set_title('Self-Attention 权重矩阵')

# 在格子中显示数值
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{attn_weights[i,j]:.2f}', ha='center', va='center', fontsize=12)

plt.colorbar(im)
plt.tight_layout()
plt.show()

print('每个 token 对其他 token 的注意力权重。')
print('权重越大（颜色越深），说明该 token 越"关注"对应的 key token。')

In [ ]:
# 解读注意力矩阵
print('=== Self-Attention 直觉理解 ===')
print()
print('注意力矩阵 A[i][j] 的含义：')
print('  Token i 在生成自己的新表示时，给 Token j 分配了多少权重')
print()
print('例如：如果 A[0][2] = 0.8，说明 Token 0 主要依赖 Token 2 的信息')
print()
print('在 NLP 中的例子：')
print('  句子: "猫 坐在 垫子 上"')
print('  "坐在" 可能会高度关注 "猫"（主语）和 "垫子"（宾语）')
print('  这就是 self-attention 捕获语义关系的方式')

---
# Part 3: IMDb 情感分类实战

现在我们把训练循环和 Transformer 结合起来，在真实的 **IMDb 电影评论数据集** 上训练一个情感分类器。

**任务**：给定一段英文电影评论，判断它是**正面**(1)还是**负面**(0)。

In [ ]:
# 加载 IMDb 数据集
from datasets import load_dataset
from transformers import AutoTokenizer

print('正在加载 IMDb 数据集...')
dataset = load_dataset('imdb')
print(f'训练集大小: {len(dataset["train"])}')
print(f'测试集大小: {len(dataset["test"])}')

In [ ]:
# 查看数据样本
sample = dataset['train'][0]
print('=== 样本展示 ===')
print(f'标签: {sample["label"]} ({"正面" if sample["label"] == 1 else "负面"})')
print(f'评论长度: {len(sample["text"])} 字符')
print(f'评论前 300 字符:\n{sample["text"][:300]}...')

# 查看标签分布
labels = dataset['train']['label']
print(f'\n标签分布: 正面={sum(labels)}, 负面={len(labels)-sum(labels)}')

In [ ]:
# Tokenizer 演示
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

text = "This movie is absolutely wonderful!"
encoded = tokenizer(text, return_tensors='pt')

print('原文:', text)
print('Token IDs:', encoded['input_ids'][0].tolist())
print('Tokens:', tokenizer.convert_ids_to_tokens(encoded['input_ids'][0]))
print('Attention Mask:', encoded['attention_mask'][0].tolist())
print(f'\n词表大小: {tokenizer.vocab_size}')

In [ ]:
# 构建 PyTorch Dataset 和 DataLoader
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 256  # 截断/padding 到 256 tokens

class IMDbDataset(Dataset):
    """将 IMDb 数据转成 PyTorch Dataset"""
    def __init__(self, hf_dataset, tokenizer, max_len):
        self.data = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        # Tokenize: 截断 + padding 到固定长度
        encoding = self.tokenizer(
            item['text'],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),       # (max_len,)
            'attention_mask': encoding['attention_mask'].squeeze(0),  # (max_len,)
            'label': torch.tensor(item['label'], dtype=torch.long)
        }

# 为了演示速度，只用前 2000 条训练、500 条测试
train_subset = dataset['train'].select(range(2000))
test_subset = dataset['test'].select(range(500))

train_dataset = IMDbDataset(train_subset, tokenizer, MAX_LEN)
test_dataset = IMDbDataset(test_subset, tokenizer, MAX_LEN)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'训练集: {len(train_dataset)} 样本, {len(train_loader)} 批次')
print(f'测试集: {len(test_dataset)} 样本, {len(test_loader)} 批次')

In [ ]:
# 检查一个 batch 的形状
batch = next(iter(train_loader))
print('Batch 数据形状:')
print(f'  input_ids:      {batch["input_ids"].shape}')       # (32, 256)
print(f'  attention_mask:  {batch["attention_mask"].shape}')  # (32, 256)
print(f'  label:           {batch["label"].shape}')           # (32,)

In [ ]:
# TransformerClassifier 模型定义
class TransformerClassifier(nn.Module):
    """
    基于 Transformer Encoder 的文本分类模型
    
    架构：
    1. Token Embedding: 将 token ID 映射为 d_model 维向量
    2. Position Embedding: 学习位置编码（不同于 sinusoidal）
    3. TransformerEncoder: 2 层, 4 头注意力
    4. 全局平均池化: 将序列表示压缩为单个向量
    5. 分类头: Linear → 2 类 (正面/负面)
    """
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes, max_len):
        super().__init__()
        # Token 嵌入：词表中每个 token 映射到 d_model 维
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # 位置嵌入：每个位置（0 到 max_len-1）有一个可学习的向量
        self.position_embedding = nn.Embedding(max_len, d_model)
        
        # Transformer Encoder 层
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,        # 嵌入维度
            nhead=nhead,            # 注意力头数
            dim_feedforward=d_model*4,  # FFN 隐藏层维度（通常 4x）
            dropout=0.1,
            activation='gelu',      # GELU 激活（GPT/BERT 标配）
            batch_first=True        # 输入形状 (batch, seq, dim)
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        
        # 分类头
        self.classifier = nn.Linear(d_model, num_classes)
        self.d_model = d_model
    
    def forward(self, input_ids, attention_mask):
        batch_size, seq_len = input_ids.shape
        
        # 1. 计算嵌入 = token embedding + position embedding
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        
        # 2. 创建 padding mask (True = 需要被忽略的 pad 位置)
        padding_mask = (attention_mask == 0)
        
        # 3. 通过 Transformer Encoder
        x = self.transformer_encoder(x, src_key_padding_mask=padding_mask)
        
        # 4. 全局平均池化（只在非 pad 位置上做）
        mask_expanded = attention_mask.unsqueeze(-1).float()  # (B, S, 1)
        x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        
        # 5. 分类
        logits = self.classifier(x)  # (B, num_classes)
        return logits

In [ ]:
# 初始化模型
VOCAB_SIZE = 30522  # bert-base-uncased 词表大小
D_MODEL = 128       # 嵌入维度（实际 BERT 用 768，这里用小模型演示）
NHEAD = 4           # 注意力头数
NUM_LAYERS = 2      # Transformer 层数
NUM_CLASSES = 2     # 二分类

classifier = TransformerClassifier(
    VOCAB_SIZE, D_MODEL, NHEAD, NUM_LAYERS, NUM_CLASSES, MAX_LEN
).to(DEVICE)

total_params = sum(p.numel() for p in classifier.parameters())
print(f'模型参数量: {total_params:,}')
print(f'设备: {DEVICE}')

In [ ]:
# 定义训练和评估函数
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-4)

def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    """训练一个 epoch: forward → loss → backward → step"""
    model.train()  # 训练模式（启用 Dropout）
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, batch in enumerate(loader):
        # 数据移到 GPU/CPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        # === 训练五步曲 ===
        logits = model(input_ids, attention_mask)    # 1. Forward
        loss = criterion(logits, labels)              # 2. Loss
        optimizer.zero_grad()                         # 3. 清零梯度
        loss.backward()                               # 4. Backward
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪
        optimizer.step()                              # 5. Update
        
        # 统计
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        if (batch_idx + 1) % 20 == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1}/{len(loader)} | '
                  f'Loss: {total_loss/(batch_idx+1):.4f} | Acc: {correct/total*100:.1f}%')
    
    return total_loss / len(loader), correct / total * 100

@torch.no_grad()  # 推理时不需要计算梯度（节省内存和计算）
def evaluate(model, loader, device):
    """在测试集上评估模型"""
    model.eval()  # 评估模式（关闭 Dropout）
    correct = 0
    total = 0
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(input_ids, attention_mask)
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return correct / total * 100

In [ ]:
# 开始训练！
import time

EPOCHS = 3
train_losses = []
val_accuracies = []

print('=' * 60)
print('开始训练 TransformerClassifier')
print('=' * 60)

for epoch in range(EPOCHS):
    start = time.time()
    
    train_loss, train_acc = train_one_epoch(
        classifier, train_loader, criterion, optimizer, DEVICE, epoch
    )
    val_acc = evaluate(classifier, test_loader, DEVICE)
    
    elapsed = time.time() - start
    train_losses.append(train_loss)
    val_accuracies.append(val_acc)
    
    print(f'\nEpoch {epoch+1}/{EPOCHS} | 耗时: {elapsed:.0f}s')
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.1f}%')
    print(f'  Val Acc:   {val_acc:.1f}%')
    print()

In [ ]:
# 训练曲线可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, EPOCHS+1), train_losses, marker='o', color='steelblue')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, EPOCHS+1), val_accuracies, marker='o', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Validation Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 模型预测演示
def predict_sentiment(text, model, tokenizer, device, max_len=256):
    """用训练好的模型预测一段文本的情感"""
    model.eval()
    encoding = tokenizer(
        text, truncation=True, padding='max_length',
        max_length=max_len, return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = F.softmax(logits, dim=-1)
        pred = logits.argmax(dim=-1).item()
    
    label = '正面' if pred == 1 else '负面'
    confidence = probs[0, pred].item()
    return label, confidence

# 测试几条评论
test_reviews = [
    "This movie is absolutely fantastic! The acting is superb and the story is gripping.",
    "Terrible film. Waste of time. The plot makes no sense at all.",
    "It was okay, nothing special but not terrible either.",
    "I loved every minute of this masterpiece. A must-watch!"
]

print('=== 模型预测演示 ===')
for review in test_reviews:
    label, conf = predict_sentiment(review, classifier, tokenizer, DEVICE)
    print(f'\n评论: "{review[:60]}..."')
    print(f'预测: {label} (置信度: {conf:.2%})')

---
# Part 4: 总结

## 今天学到了什么

1. **训练循环五步曲**：`Forward → Loss → zero_grad → Backward → Step`
2. **Self-Attention**：通过 Q, K, V 计算注意力权重，让每个 token 能"看到"其他 token
3. **完整实战**：从数据加载 → Tokenizer → Dataset → 模型定义 → 训练 → 评估 → 预测

## 关键概念对照表

| 概念 | 作用 | 类比 |
|------|------|------|
| Forward Pass | 数据流过网络得到预测 | 考试答题 |
| Loss | 衡量预测与真实的差距 | 批改试卷 |
| Backward Pass | 计算每个参数对 loss 的贡献 | 分析错因 |
| Optimizer Step | 调整参数减小 loss | 针对性复习 |
| Self-Attention | 让 token 之间交换信息 | 小组讨论 |

## 思考题

1. **为什么每步都要 `optimizer.zero_grad()`？** 如果不清零会怎样？在什么场景下我们反而*想要*梯度累积？

2. **`model.train()` 和 `model.eval()` 到底改变了什么？** 提示：想想 Dropout 和 BatchNorm 的行为差异。

3. **如果把 `MAX_LEN` 从 256 改成 512，训练时间和显存会怎么变化？** Self-attention 的计算复杂度是多少？

4. **挑战题**：尝试修改模型，用 `[CLS]` token 的输出代替全局平均池化来做分类，比较效果。

## 下一步

Day 2 我们将从**分类**转向**生成**，从零构建一个 GPT 模型，训练它生成莎士比亚风格的文本！